In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import FuncFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm
# from scipy.interpolate import interp1d  
from scipy import stats
from matplotlib.ticker import LogLocator
from matplotlib.backends.backend_pdf import PdfPages

import pandas as pd

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def LimitTrackedArraysRows(trackedArrays, limit=None): #limit=(0,70000)
    if limit is None:
        return trackedArrays
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            trackedArrays[parcelType][parcelDepth] \
            = trackedArrays[parcelType][parcelDepth][limit[0]:limit[1], :]
    return trackedArrays

In [ ]:
def GetData(tRange, pValues=np.arange(10000)):
    t1 = calculate_timestep(tRange[0])
    t2 = calculate_timestep(tRange[1])

    varNames = ['Z','Y','X'] + ['QCQI','QV','W']
    dtypes = {'Z': np.int16, 
              'Y': np.int16, 
              'X': np.int16, 
              'QCQI': np.float32, 
              'QV': np.float32,
               'W': np.float32}
    
    Nt = t2 - t1 + 1
    Np = len(pValues)
    dataDictionary = {var: np.zeros((Nt, Np), dtype=dtypes[var]) for var in varNames}
    dataDictionary['tSteps'] = np.arange(t1, t2+1)
    dataDictionary['pValues'] = pValues
    
    for i, t in enumerate(tqdm(dataDictionary['tSteps'])):
        for varName in varNames:
            raw = CallLagrangianArray(ModelData, DataManager, ModelData.timeStrings[t], varName)[pValues]
            dataDictionary[varName][i, :] = np.round(raw) if varName in ('Z','Y','X') else raw
            
    return dataDictionary

def calculate_timestep(time_hr=12):
    return np.abs(ModelData.time_hrs-time_hr).argmin()

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
#Find Detraining Parcels at MidLevels

def DetectDetrainment(dataDictionary, zLimit=(2,4), cloudThreshold=1e-5):
    zLevel = ModelData.zh[dataDictionary['Z']]
    QC = dataDictionary['QCQI']  # using qcqi for now

    inLayer = (zLevel >= zLimit[0]) & (zLevel <= zLimit[1])
    isCloudy = QC > cloudThreshold
    isEnvironment = ~isCloudy

    wasCloudy = np.zeros_like(isCloudy)
    wasCloudy[1:] = isCloudy[:-1]  # shift cloudy at t-1 boolean to time t

    movedGridbox = DetectGridboxCross(dataDictionary)

    isDetrain = wasCloudy & isEnvironment & inLayer & movedGridbox
    return isDetrain

def DetectGridboxCross(dataDictionary):
    X = dataDictionary['X']
    Y = dataDictionary['Y']

    xChanged = np.zeros_like(X, dtype=bool)
    yChanged = np.zeros_like(Y, dtype=bool)
    xChanged[1:] = X[1:] != X[:-1]
    yChanged[1:] = Y[1:] != Y[:-1]

    return xChanged | yChanged

In [ ]:
#Track Ascending Surface Parcels

def DetectUpdraftEvents(dataDictionary, wThresh=0.1,cloudThreshold=1e-5, 
                        minAscentTime_mins=20,minCloudTime_mins=5, 
                        minHeightGain=1.0, surfaceZLimit=0.5):
    W  = dataDictionary['W']
    QC = dataDictionary['QCQI']
    Z  = ModelData.zh[dataDictionary['Z']]
    Nt = Z.shape[0]
    
    ascentWindow = MinutesToTimesteps(minAscentTime_mins)
    minCloudSteps=MinutesToTimesteps(minCloudTime_mins)
    
    sustainedUpdraft = RollingAllTrue(W > wThresh, ascentWindow)          # w>thresh whole window
    cloudSteps = RollingCountTrue(QC > cloudThreshold, ascentWindow)      # cloudy for enough of it
    heightGain = Z[ascentWindow-1:] - Z[:Nt-ascentWindow+1]               # actually rose

    n = sustainedUpdraft.shape[0] # target length for the shifted checks below
    
    # only parcels that initiate below 0.5 km
    wasSurfaceParcel = np.zeros_like(sustainedUpdraft)
    wasSurfaceParcel[1:] = Z[:n-1] <= surfaceZLimit

    # make sure parcel undergoes acceleration (w failed threshold the step before)
    notUpdraftBefore = np.zeros_like(sustainedUpdraft)
    notUpdraftBefore[1:] = ~(W[:n-1] > wThresh)

    isUpdraftEvent_ascentRelative = (
        sustainedUpdraft
        & (cloudSteps >= minCloudSteps)
        & wasSurfaceParcel
        & notUpdraftBefore
        & (heightGain >= minHeightGain)
    )
    return isUpdraftEvent_ascentRelative   # shape (Nt-ascentWindow, Np); True at t = event starting at t

def MinutesToTimesteps(minutes):
    """Converts a duration in minutes to a number of model timesteps."""
    return max(1, round((minutes * 60) / ModelData.dt))

def RollingAllTrue(cond, window):
    """True at row t if cond is True for every step in t ... t + window-1."""
    c = np.cumsum(np.vstack([np.zeros((1, cond.shape[1])), cond]), axis=0)
    return (c[window:] - c[:-window]) == window

def RollingCountTrue(cond, window):
    """Count of True steps within t ... t + window-1."""
    c = np.cumsum(np.vstack([np.zeros((1, cond.shape[1])), cond]), axis=0)
    return c[window:] - c[:-window]

####################################################################################

def PadToFullNt(eventArray, Nt, fillValue=False):
    Np = eventArray.shape[1]
    nMissing = Nt - eventArray.shape[0]
    pad = np.full((nMissing, Np), fillValue, dtype=eventArray.dtype)
    return np.vstack([eventArray, pad])

####################################################################################
#Get Full-Duration Ascent Mask
def GetAscentDurationMask(dataDictionary, isUpdraftEvent, wThresh=0.1):
    """
    Given confirmed ascent-start flags (isUpdraftEvent) and the raw w>wThresh
    condition, find the true full-duration run each confirmed start belongs
    to, and return a (Nt, Np) mask that's True for the entire ascent span
    (not just the start timestep).
    """
    W = dataDictionary['W']
    Nt, Np = W.shape

    rawUpdraft = W > wThresh
    runStart, runEnd, runParcel = FindRuns(rawUpdraft)

    evT, evP = np.where(isUpdraftEvent)

    # match confirmed event starts to their corresponding raw run
    eventDF = pd.DataFrame({'t': evT, 'p': evP})
    runDF = pd.DataFrame({'t': runStart, 'end': runEnd, 'p': runParcel})
    matched = eventDF.merge(runDF, on=['t', 'p'], how='inner')

    isAscending = BuildFullDurationMask(
        matched['t'].values, matched['end'].values, matched['p'].values, Nt, Np
    )
    return isAscending

def FindRuns(condition):
    Nt, Np = condition.shape
    padded = np.zeros((Nt + 2, Np), dtype=np.int8)
    padded[1:-1, :] = condition
    d = np.diff(padded, axis=0)
    startRows, startCols = np.where(d == 1)
    endRows, endCols = np.where(d == -1)
    orderS = np.lexsort((startRows, startCols))
    orderE = np.lexsort((endRows, endCols))
    startRows, startCols = startRows[orderS], startCols[orderS]
    endRows = endRows[orderE]
    return startRows, endRows, startCols

def BuildFullDurationMask(startIdx, endIdx, parcelIdx, Nt, Np):
    delta = np.zeros((Nt + 1, Np), dtype=np.int8)
    np.add.at(delta, (startIdx, parcelIdx), 1)
    np.add.at(delta, (np.clip(endIdx, 0, Nt), parcelIdx), -1)
    return np.cumsum(delta[:-1], axis=0) > 0

####################################################################################

def PlotAscentValidation(dataDictionary, isAscending, p, wThresh=0.1, cloudThreshold=1e-5*1e3):
    W = dataDictionary['W'][:, p]
    QC = dataDictionary['QCQI'][:, p]*1e3
    tSteps = dataDictionary['tSteps']
    timeOfDay = ModelData.time_hrs[tSteps]

    fig, ax1 = plt.subplots(figsize=(10, 4))

    ax1.plot(timeOfDay, W, color='black', lw=1.5, label='w')
    ax1.axhline(wThresh, color='gray', ls='--', lw=1, label=f'wThresh={wThresh}')
    ax1.set_xlabel('time of day')
    ax1.set_ylabel('w (m/s)')
    ax1.xaxis.set_major_formatter(FuncFormatter(HoursToHHMM))

    mask = isAscending[:, p]

    # duration of the ascending region, from the actual masked timesteps
    if mask.any():
        durMin = round(mask.sum() * ModelData.dt / 60)
        ascentLabel = f'isAscending ({durMin} min)'
    else:
        ascentLabel = 'isAscending (none)'

    ax1.fill_between(timeOfDay, ax1.get_ylim()[0], ax1.get_ylim()[1], where=mask,
                      color='tab:orange', alpha=0.15, step='mid', label=ascentLabel)

    ax2 = ax1.twinx()
    ax2.plot(timeOfDay, QC, color='blue', lw=1.5, label='QCQI')
    ax2.axhline(cloudThreshold, color='blue', ls=':', lw=1, alpha=0.6,
                label=f'cloudThreshold={cloudThreshold}')
    ax2.set_ylabel('QCQI (cloud water+ice)', color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

    ax1.set_title(f'Parcel {p} — w and QCQI profile vs detected ascent')
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

def HoursToHHMM(hrs, pos=None):
    h = int(hrs)
    m = int(round((hrs - h) * 60))
    if m == 60:
        h += 1
        m = 0
    return f"{h:02d}:{m:02d}"

In [ ]:
#Find Ascending-Parcel Intersections with Detrainment Events

def FindIntersections(dataDictionary, isAscending, isDetrain, windowMin=30):
    cellId = AssignCellId(dataDictionary)
    Nt, Np = isAscending.shape
    windowSteps = MinutesToTimesteps(windowMin)

    # detrainment events: where and when each one happened
    dT, dP = np.where(isDetrain)
    dCell = cellId[dT, dP]

    # stretch each event forward so it stays "active" for windowMin minutes
    offsets = np.arange(windowSteps)
    expT = (dT[:, None] + offsets[None, :]).ravel()
    expCell = np.repeat(dCell, windowSteps)
    expDetrainParcel = np.repeat(dP, windowSteps)
    valid = expT < Nt
    detrainDF = pd.DataFrame({
        't': expT[valid], 'cellId': expCell[valid], 'detrain_parcel': expDetrainParcel[valid]
    })

    # ascending parcels: where each one is at every timestep
    aT, aP = np.where(isAscending)
    aCell = cellId[aT, aP]
    ascendDF = pd.DataFrame({'t': aT, 'cellId': aCell, 'ascend_parcel': aP})

    # match on same gridbox + same time
    matches = ascendDF.merge(detrainDF, on=['t', 'cellId'], how='inner')
    matches = matches[matches['ascend_parcel'] != matches['detrain_parcel']]  # drop self-matches

    return matches

def AssignCellId(dataDictionary):
    """
    Assign unique gridcell IDs 
    using linear bijective index mapping
    """
    dims = GetGridDims()
    X = dataDictionary['X']; Y = dataDictionary['Y']; Z = dataDictionary['Z']
    return np.ravel_multi_index((X, Y, Z), dims)

def InverseCellId(cellId):
    """
    Converts unique gridcell IDs u
    sing linear bijective index mapping 
    back into X, Y, Z coordinates.
    """
    dims = GetGridDims()
    return np.unravel_index(cellId, dims)

def GetGridDims():
    return (ModelData.Nxh + 1, ModelData.Nyh + 1, ModelData.Nzh + 1)
    
####################################################################################


import matplotlib.patheffects as pe

OUTLINE = [pe.withStroke(linewidth=3, foreground='white')]

def PlotIntersection(dataDictionary, intersections, row=0, dx_km=1.0,
                      xlim=(300, 320), ylim=(0, 5), ax=None, tOverride=None,
                      plotVar='w', plotType='contourf', nContours=15):
    cfg = VAR_CONFIG[plotVar]

    rec = intersections.iloc[row]
    tCenter = int(rec['t'])
    t = tCenter if tOverride is None else tOverride
    cellId = int(rec['cellId'])
    ascendP = int(rec['ascend_parcel'])
    detrainP = int(rec['detrain_parcel'])
    ix, iy, iz = InverseCellId(cellId)
    Nt = dataDictionary['Z'].shape[0]
    t = max(0, min(t, Nt - 1))
    tStepAbs = dataDictionary['tSteps'][t]

    field = CallVariable(ModelData, DataManager, ModelData.timeStrings[tStepAbs], cfg['varname'])
    fieldSlice = field[:, iy, :] * cfg['scale']

    nz, nx = fieldSlice.shape
    xGrid = np.arange(nx) * dx_km
    zGrid = ModelData.zh[:nz]

    ownFig = ax is None
    if ownFig:
        fig, ax = plt.subplots(figsize=(8, 5))

    if plotType == 'pcolormesh':
        im = ax.pcolormesh(xGrid, zGrid, fieldSlice, cmap=cfg['cmap'],
                            vmin=cfg['vmin'], vmax=cfg['vmax'], shading='auto')
    else:
        levels = np.linspace(cfg['vmin'], cfg['vmax'], nContours)
        im = ax.contourf(xGrid, zGrid, fieldSlice, levels=levels, cmap=cfg['cmap'], extend='both')

    xA = dataDictionary['X'][:t+1, ascendP] * dx_km
    zA = ModelData.zh[dataDictionary['Z'][:t+1, ascendP]]
    ax.plot(xA, zA, color='black', lw=1.5, marker='o', ms=3, label=f'ascending parcel {ascendP}',
            path_effects=OUTLINE)
    ax.scatter(xA[-1], zA[-1], color='lime', s=90, edgecolor='black', zorder=5,
               label='ascending parcel')

    dT_events, dP_events = np.where(isDetrain)
    detrainT = dT_events[(dP_events == detrainP)][0]
    tD = min(t, detrainT)
    xD = dataDictionary['X'][:tD+1, detrainP] * dx_km
    zD = ModelData.zh[dataDictionary['Z'][:tD+1, detrainP]]
    ax.plot(xD, zD, color='blue', lw=1.5, marker='s', ms=3, label=f'detraining parcel {detrainP}',
            path_effects=OUTLINE)
    ax.scatter(xD[-1], zD[-1], color='magenta', s=90, edgecolor='black', zorder=5,
               label='detrainment location')

    ax.set_xlabel('x (km)')
    ax.set_ylabel('z (km)')
    timeLabel = HoursToHHMM(ModelData.time_hrs[tStepAbs])
    tag = f't={timeLabel}' + (' (intersection)' if t == tCenter else '')
    ax.set_title(f'{plotVar} slice at y={iy*dx_km:.1f} km, {tag}')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    if ownFig:
        plt.colorbar(im, ax=ax, label=cfg['label'])
        ax.legend(fontsize=8, loc='best')
        plt.tight_layout()
        plt.show()
    return im

VAR_CONFIG = {
    'w':  {'varname': 'winterp', 'cmap': 'RdBu_r', 'vmin': -5,  'vmax': 5,   'label': 'w (m/s)',
           'scale': 1.0, 'maskBelow': None},
    'qc': {'varname': 'qcqi',    'cmap': 'turbo',  'vmin': 0.01, 'vmax': 2,  'label': 'QCQI (g/kg)',
           'scale': 1e3, 'maskBelow': 0.01},   # values below this become NaN -> transparent
}

def PlotIntersectionTimeSeries(dataDictionary, intersections, row=0, tOffsets=np.arange(-12,6,2),
                                 ncols=3, plotVar='w', **kwargs):
    cfg = VAR_CONFIG[plotVar]
    rec = intersections.iloc[row]
    tCenter = int(rec['t'])
    Nt = dataDictionary['Z'].shape[0]
    tList = [tCenter + off for off in tOffsets if 0 <= tCenter + off < Nt]
    nPanels = len(tList)
    nrows = int(np.ceil(nPanels / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 5*nrows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    im = None
    for ax, t in zip(axes, tList):
        im = PlotIntersection(dataDictionary, intersections, row=row, ax=ax, tOverride=t,
                               plotVar=plotVar, **kwargs)

    for ax in axes[nPanels:]:
        ax.axis('off')

    fig.colorbar(im, ax=axes[:nPanels], label=cfg['label'], shrink=0.6)
    axes[0].legend(fontsize=8, loc='best')
    plt.show()

In [ ]:
##############################################
#COMPUTING

In [ ]:
dataDictionary=GetData(tRange=[12,13],pValues=np.arange(100_000))

In [ ]:
#Find Detraining Parcels at MidLevels
isDetrain=DetectDetrainment(dataDictionary)

In [ ]:
#Track Ascending Surface Parcels
isUpdraftEvent_ascentRelative = DetectUpdraftEvents(dataDictionary)
isUpdraftEvent = PadToFullNt(isUpdraftEvent_ascentRelative, Nt=dataDictionary['Z'].shape[0])
isAscending = GetAscentDurationMask(dataDictionary, isUpdraftEvent)

#testing
examples = np.where(isAscending)[1]
PlotAscentValidation(dataDictionary, isAscending, p=examples[5])

In [ ]:
#Find Ascending-Parcel Intersections with Detrainment Events
intersections = FindIntersections(dataDictionary, isAscending, isDetrain)
print(intersections)

#testing
PlotIntersectionTimeSeries(dataDictionary, intersections, row=0, plotVar='w')
PlotIntersectionTimeSeries(dataDictionary, intersections, row=0, plotVar='qc')